# Analyse der Datensatzgröße & Datenbereinigung

## 1. Ursachen für die Reduzierung der Datenmenge auf 13.676 Einträge

Die Reduzierung des ursprünglichen "MovieLens 25M"-Datensatzes (der ca. 62.000 Filme umfasst) auf exakt **13.676 Filme** im bereinigten Datensatz resultiert aus notwendigen Filterungsschritten in der Daten-Pipeline:

- **Verfügbarkeit des Tag-Genomes**: Die für das Mood-Training essenziellen Stimmungs-Vektoren (das Tag-Genome) sind im MovieLens-Datensatz nur für knapp 14.000 Filme berechnet worden. Filme ohne diese Vektoren besitzen keine Labels und mussten über einen `inner join` entfernt werden.
- **Ausschluss fehlender Filmbeschreibungen**: Da das Modell lernen soll, aus Freitext Stimmungen vorherzusagen, ist eine Textbeschreibung zwingend erforderlich. Filme, für die aus TMDB keine Beschreibung (`overview`) geladen werden konnte, wurden herausgefiltert.

## 2. Eignung der verbleibenden Datenmenge für das Fine-Tuning

Obwohl 13.676 Datensätze für das Training eines neuronalen Netzes von Grund auf unzureichend wären, ist diese Menge für den gewählten Fine-Tuning-Ansatz gut geeignet:

- **Transfer Learning**: Das Projekt nutzt ein vortrainiertes Transformer-Modell (z. B. DistilBERT), das bereits über umfassendes sprachliches Vorwissen verfügt.
- **Optimaler Bereich (Sweet Spot)**: Für das Fine-Tuning eines solchen Modells gilt eine dichte, qualitativ hochwertige Datenbasis von 10.000 bis 20.000 Beispielen als ideal. Sie bietet ausreichend Varianz für robustes Lernen bei gleichzeitig sehr kurzen Trainingszeiten und minimalem Rauschen.

HuggingFace-Modell wurde hochgeladen. Lokales Training erfolgt ueber `uv run --no-sync python -m src.training.train_model`.


# Trainings- und Optimierungsnotizen

Dieser Abschnitt dokumentiert die Entwicklung der Trainingspipeline chronologisch: Ausgangslage, technische Anpassungen, Experimentreihen, aktuelles Ergebnis und die verbleibenden sinnvollen To-dos.

## 1. Ausgangslage: Daten und Label-Herkunft

Die Trainingsdaten entstehen aus einer Verknuepfung von TMDB-Filmbeschreibungen (`overview`) und MovieLens-Tag-Genome-Daten. `src/data/prepare_data.py` erzeugt daraus `cleaned_movie_data.csv` und `mood_tags.json`.

Aktuell werden die 100 haeufigsten MovieLens-Genome-Tags ausgewaehlt, deren Relevanz mindestens `Settings.DECISION_THRESHOLD` erreicht. Diese Auswahl ist technisch nachvollziehbar, aber semantisch nicht streng auf Stimmungen begrenzt.

Dabei wurde sichtbar, dass `mood_tags.json` neben echten Mood-/Tone-Tags auch Genres, Plot-Merkmale, Produktionskontext, Auszeichnungen und formale Eigenschaften enthaelt, z. B. `action`, `oscar`, `pg-13`, `based on book`, `criterion`, `plot` oder `story`.

Damit trainiert das Modell derzeit eher einen Top-100-MovieLens-Tag-Predictor als einen reinen Mood-Predictor. Fuer den Prototyp ist das brauchbar, die spaetere Empfehlungsqualitaet haengt jedoch stark von einer saubereren Label-Liste ab.


## 2. Laufzeitproblem: Sequenzlaenge und Padding

Fuer den Bot wurde `MAX_SEQUENCE_LENGTH = 512` gesetzt, damit laengere User-Anfragen nicht zu frueh abgeschnitten werden. Beim Training war das zunaechst teuer, weil die Tokenisierung mit `padding="max_length"` jedes Beispiel auf 512 Tokens aufgefuellt hat.

Die gemessenen Tokenlaengen der aktuellen Trainingsdaten zeigen, dass diese fixe Auffuellung kaum Nutzen bringt:

- Mittelwert: ca. 62 Tokens
- Median: ca. 57 Tokens
- 95. Perzentil: ca. 119 Tokens
- 99. Perzentil: ca. 163 Tokens
- Maximum: ca. 235 Tokens
- Beispiele ueber 256 Tokens: 0

Anpassung: `dataset.py` tokenisiert ohne fixes Padding. `train_model.py` nutzt `DataCollatorWithPadding`, sodass pro Batch nur bis zur laengsten Sequenz im Batch aufgefuellt wird. Damit bleiben lange Bot-Eingaben erlaubt, ohne kurze Trainingsbeispiele unnoetig teuer zu machen.


## 3. Ausfuehrung auf der CUDA-Maschine

Beim Training wurde beobachtet, dass `uv run` die Umgebung anhand von `uv.lock` wieder auf die CPU-Version von PyTorch synchronisieren kann. Dadurch kann ein zuvor manuell installiertes CUDA-PyTorch wieder ueberschrieben werden.

Auf der CUDA-Maschine wird deshalb zuerst synchronisiert, danach die passende Torch-Version installiert und anschliessend mit `--no-sync` trainiert:

```powershell
uv sync
uv pip install torch --torch-backend=cu130 --reinstall
uv run --no-sync python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
```

Das Script `run_selected_experiments.sh` verwendet intern bereits `uv run --no-sync`, damit diese Umgebung waehrend der Experimentreihe stabil bleibt.


## 4. Erste Hyperparameter-Beobachtungen (vor Pipeline-Optimierungen)

Die erste grobe Experimentreihe zeigte, dass kleinere Batch Sizes deutlich besser funktionieren. Die Werte beziehen sich auf `eval_macro_f1` mit fixem Threshold `0.5`.

| Run | Bester eval_macro_f1 |
| --- | ---: |
| epochs=5, batch_size=8, lr=2e-5 | 0.291093 |
| epochs=5, batch_size=16, lr=2e-5 | 0.222505 |
| epochs=5, batch_size=24, lr=2e-5 | 0.207173 |
| epochs=5, batch_size=32, lr=2e-5 | 0.167727 |

Daraufhin wurde der Suchraum auf `batch_size=8` fokussiert. Die zweite ausgewaehlte Experimentreihe ergab vor den spaeteren Optimierungen:

| Run | Bester eval_macro_f1 | Bemerkung |
| --- | ---: | --- |
| epochs=8, batch_size=8, lr=2e-5 | 0.337729 | bester beobachteter Run vor den neuen Loss-/Threshold-Aenderungen |
| epochs=5, batch_size=8, lr=3e-5 | 0.316420 | liegt aktuell als Baseline-Metadatenwert in `final_model/best_model_info.json` |
| epochs=6, batch_size=8, lr=2e-5 | 0.302998 | besser als 5 Epochen bei gleicher LR |
| epochs=5, batch_size=8, lr=2e-5 | 0.277513 | solide, aber unter 6/8 Epochen |
| epochs=5, batch_size=8, lr=1e-5 | 0.193982 | lernte zu langsam |

Aus diesen Runs ergab sich `batch_size=8` als bevorzugte Batch Size. Mehr Epochen halfen bis mindestens Epoche 6. `1e-5` lernte zu langsam; `3e-5` war bei 5 Epochen besser als `2e-5`. Vor den spaeteren Code-Optimierungen war `epochs=8, batch_size=8, lr=2e-5` der beste beobachtete Run.


## 5. Anpassungen an der Trainingspipeline

Danach wurde die Trainingspipeline verbessert, ohne die Label-Liste neu zu kuratieren:

1. **Learning Rate als CLI-Parameter**  
   `train_model.py` akzeptiert `--learning_rate`, damit Experimente ohne Codeaenderung vergleichbar laufen.

2. **Dynamic Padding**  
   `dataset.py` tokenisiert ohne `padding="max_length"`; `train_model.py` nutzt `DataCollatorWithPadding`.

3. **Gewichtete Loss-Funktion**  
   `WeightedTrainer` verwendet `BCEWithLogitsLoss(pos_weight=...)`, damit seltenere positive Labels staerker beruecksichtigt werden. Ziel ist eine bessere Macro-F1 ueber alle Labels, nicht nur ueber haeufige Tags.

4. **Metriken mit Threshold-Tracking**  
   `macro_f1` bleibt der Score beim festen Threshold `0.5`. Zusaetzlich scannt `compute_metrics()` globale Thresholds von `0.1` bis `0.9` und loggt den besten Wert als `macro_f1_best_threshold` sowie den zugehoerigen `decision_threshold`.

5. **Zentrale Trainingskonfiguration**  
   Wichtige Stellschrauben wie Default-Epochen, Batch Size, Learning Rate, Test-Split, Warmup, Weight Decay, Checkpoint-Limit, Threshold-Suchbereich und `Settings.RANDOM_SEED` liegen jetzt in `Settings`. Der Seed steuert sowohl den Dataset-Split als auch `seed` und `data_seed` in `TrainingArguments`.

6. **Final-Model-Vergleich**  
   `final_model` wird nicht mehr nach jedem Run blind ueberschrieben. Stattdessen vergleicht `training_utils.py` den neuen besten `macro_f1` mit `final_model/best_model_info.json` und speichert nur bei Verbesserung.

Da `metric_for_best_model="macro_f1"` gesetzt ist, wird das finale Modell weiterhin nach dem festen Threshold `0.5` ausgewaehlt. Der optimierte Threshold ist aktuell ein Analysewert, keine Bot-Laufzeitlogik.


## 6. Ausgewaehlte Experimentreihe (nach Pipeline-Optimierungen)

Nach den Pipeline-Anpassungen wurde der Suchraum gezielt um die bisher besten Parameter gelegt. Das Script `run_selected_experiments.sh` fuehrt diese Runs aus:

```text
epochs=10, batch_size=8, lr=2e-5
epochs=12, batch_size=8, lr=2e-5
epochs=8,  batch_size=8, lr=3e-5
epochs=10, batch_size=8, lr=3e-5
epochs=8,  batch_size=4, lr=2e-5
```

Startbefehl unter PowerShell:

```powershell
& "C:\\Program Files\\Git\\bin\\bash.exe" "./run_selected_experiments.sh"
```

Die wichtigsten TensorBoard-Kurven fuer die Auswertung sind:

- `eval/macro_f1`: Entscheidende Vergleichsmetrik bei Threshold `0.5`
- `eval/macro_f1_best_threshold`: bester F1 nach globalem Threshold-Scan
- `eval/decision_threshold`: bester gescannter Threshold
- `eval/roc_auc`: Rangordnungsqualitaet der Wahrscheinlichkeiten
- `eval/loss` und `train/loss`: Trainingsverlauf und Overfitting-Signale


## 7. Ergebnis der optimierten Experimentreihe

Die ausgewaehlte Experimentreihe ist vollstaendig durchgelaufen. Alle Runs haben TensorBoard-Scalars geschrieben, und `final_model` wurde auf den besten Run aktualisiert.

Bestes gespeichertes Modell laut `src/training/models/final_model/best_model_info.json`:

```json
{
    "best_macro_f1": 0.45365440070513957,
    "best_macro_f1_best_threshold": 0.45365440070513957,
    "best_decision_threshold": 0.5000000000000001,
    "metric_for_best_model": "macro_f1",
    "run_name": "run_20260530-165150-epochs8_bs8_lr3e-05",
    "epochs": 8,
    "batch_size": 8,
    "learning_rate": 3e-05,
    "best_global_step": 12312
}
```

Ranking der relevanten neuen Runs nach bestem `eval/macro_f1`:

| Rang | Run | Best macro_f1 | Threshold | ROC-AUC | Bemerkung |
| ---: | --- | ---: | ---: | ---: | --- |
| 1 | epochs=8, batch_size=8, lr=3e-5 | 0.453654 | 0.50 | 0.760650 | aktuelles `final_model` |
| 2 | epochs=10, batch_size=8, lr=3e-5 | 0.452060 | 0.50 | 0.760882 | fast gleich gut |
| 3 | epochs=10, batch_size=8, lr=2e-5 | 0.451697 | 0.50 | 0.759121 | fast gleich gut |
| 4 | epochs=8, batch_size=4, lr=2e-5 | 0.451503 | 0.50 | 0.757287 | ebenfalls nah dran, aber laengere Laufzeit |
| 5 | epochs=12, batch_size=8, lr=2e-5 | 0.450305 | 0.45 | 0.757866 | etwas schwaecher |

Beobachtungen:

- Der beste beobachtete `macro_f1` stieg von ca. `0.338` auf `0.454`.
- Die Top-Runs liegen sehr nah beieinander; `epochs=8, batch_size=8, lr=3e-5` ist aber zugleich der beste und einer der kuerzeren Runs.
- Der beste Threshold liegt beim Gewinner bei `0.50`. Der Score haengt damit nicht von einem extrem niedrigen oder hohen Threshold ab.
- `ROC-AUC` liegt bei den Top-Runs um `0.76`. Positive Beispiele werden damit deutlich besser gerankt als durch Zufall, auch wenn die absolute F1 noch Raum nach oben laesst.
- `eval_loss` steigt bei einigen Runs, waehrend `macro_f1` steigt. Die Modellwahl erfolgt deshalb nach der Zielmetrik und nach qualitativer Bot-Pruefung, nicht nur nach Loss.


## 8. Breite Grid-Suche mit `run_experiments.sh`

Nach der ausgewaehlten Experimentreihe wurde `run_experiments.sh` erneut ausgefuehrt. Das Script testet eine breitere Kombination aus Epochen und Batch Sizes bei `learning_rate=2e-5`:

```text
epochs in [5, 8, 10, 12]
batch_size in [8, 16, 24, 32]
learning_rate = 2e-5
```

Die Ausfuehrung nutzt jetzt ebenfalls `uv run --no-sync`, damit die CUDA-PyTorch-Installation auf der Trainingsmaschine nicht durch einen erneuten Sync ersetzt wird.

Ranking der besten Runs aus dieser Grid-Suche nach `eval/macro_f1`:

| Rang | Run | Best macro_f1 | Threshold | ROC-AUC | Bemerkung |
| ---: | --- | ---: | ---: | ---: | --- |
| 1 | epochs=12, batch_size=8, lr=2e-5 | 0.451165 | 0.50 | 0.758470 | bester Run dieser Grid-Suche |
| 2 | epochs=5, batch_size=8, lr=2e-5 | 0.450603 | 0.50 | 0.759213 | sehr nah am besten Grid-Run |
| 3 | epochs=10, batch_size=8, lr=2e-5 | 0.449669 | 0.50 | 0.759020 | etwas schwaecher |
| 4 | epochs=8, batch_size=8, lr=2e-5 | 0.448986 | 0.50 | 0.757398 | schwaecher als der fruehere 8/8/3e-5 Run |
| 5 | epochs=10, batch_size=16, lr=2e-5 | 0.447697 | 0.50 | 0.756973 | groessere Batch Size bleibt schwaecher |

Einordnung:

- Die breite Grid-Suche bestaetigt `batch_size=8` als beste Batch Size im getesteten Bereich.
- Die beste Konfiguration dieser Grid-Suche erreicht `macro_f1=0.451165` und schlaegt den vorherigen besten Run `epochs=8, batch_size=8, lr=3e-5` mit `macro_f1=0.453654` nicht.
- Die Ergebnisse stuetzen damit die vorherige Einordnung: Eine weitere breite Suche bei `lr=2e-5` bringt weniger Nutzen als gezielte Tests um `batch_size=8` und `lr=3e-5`.
- Das aktuell lokal gespeicherte `final_model` wurde durch diese Grid-Suche auf `run_20260530-184925-epochs12_bs8_lr2e-05` gesetzt. Dieses Modell ist besser als die anderen Grid-Runs, aber schlechter als der zuvor dokumentierte beste 8/8/3e-5-Run.


## 9. Aktueller Stand und Einordnung

Technisch funktioniert die Trainingspipeline jetzt durchgaengig:

- Daten werden geladen und dynamisch gepaddet.
- Das Training nutzt eine gewichtete BCE-Loss fuer unausgeglichene Labels.
- Experimente koennen per CLI und Script gestartet werden; zentrale Trainingsdefaults und Seed liegen in `Settings`.
- Das beste Modell wird nicht mehr versehentlich durch schlechtere Runs ueberschrieben.
- Das aktuelle `final_model` laedt erfolgreich in der Inferenz und erzeugt Empfehlungen.

Fachlich bleibt aber die Label-Definition der groesste Hebel. Ein guter `macro_f1` auf den aktuellen Labels bedeutet, dass das Modell die ausgewaehlten MovieLens-Genome-Tags besser trifft. Es bedeutet noch nicht automatisch, dass die Filmempfehlungen nach menschlichem Mood-Verstaendnis optimal sind.

Der Bot nutzt aktuell keine binaeren Tag-Entscheidungen, sondern die Wahrscheinlichkeitsvektoren des Modells und vergleicht sie per Cosine Similarity mit den Filmvektoren. Der getrackte Threshold dient daher der Trainingsanalyse, ist aber nicht direkt Teil der Bot-Logik.

Aus dem aktuellen Stand folgt: Weitere breite Hyperparameter-Suchen haben wahrscheinlich weniger Nutzen als eine qualitative Bot-Evaluation und eine bessere Mood-Tag-Definition. Falls das beste bisher beobachtete Modell lokal wieder als `final_model` liegen soll, muss `epochs=8, batch_size=8, learning_rate=3e-5` erneut trainiert werden.


## 10. Bot-Handling und Mistral-Chat-Schicht

Nach der Trainings- und Recommender-Arbeit wurde das Telegram-Bot-Handling ueberarbeitet. Wichtig war dabei, die bestehende Empfehlungslogik nicht zu ersetzen: `MoodPredictor` und `MovieRecommender` bleiben fuer die Filmauswahl verantwortlich. Mistral wird nur als Chat-Schicht davor und als Antwort-Schicht danach genutzt.

Chronologie der Aenderungen:

1. **Problem im Bot-Verhalten identifiziert**  
   Der Bot hat jede Textnachricht direkt an `predictor.predict(...)` uebergeben. Dadurch erzeugten Eingaben wie `+`, `#`, `gre` oder `erb` trotzdem einen Wahrscheinlichkeitsvektor. Da der Recommender immer die naechsten Filme per Cosine Similarity findet, entstanden scheinbar gueltige Empfehlungen fuer inhaltlich ungueltige Eingaben.

2. **Erster lokaler Eingabe-Guard eingebaut**  
   In `src/telegram_bot/bot.py` wurden `normalize_user_message(...)` und `is_valid_movie_request(...)` ergaenzt. Damit wurden leere Nachrichten und reine Symbol-Eingaben frueh abgefangen. Dieser Guard war bewusst einfach und spaeter vor allem als Fallback gedacht.

3. **Grenze des bestehenden Modells geklaert**  
   Das trainierte Modell ist ein Tag-/Mood-Predictor. Es besitzt keine Klasse fuer `invalid_request`, `not_movie_related` oder `nonsense`. Deshalb kann es nicht verlaesslich selbst entscheiden, ob eine Anfrage sinnvoll ist. Die hohen Prozentwerte im Bot sind keine Modell-Confidence, sondern Aehnlichkeitswerte zwischen User-Vektor und Filmvektoren.

4. **Mistral als ChatController eingefuehrt**  
   Mit `src/telegram_bot/chat_controller.py` wurde eine separate Controller-Schicht eingefuehrt. Sie ruft die Mistral Chat-Completions API ueber die Standardbibliothek auf und liefert eine strukturierte Entscheidung mit festen Actions: `recommend`, `clarify`, `reject`, `help` und `smalltalk`.

5. **Strukturierte Entscheidungen statt freier LLM-Antworten**  
   Der Controller nutzt `response_format` mit JSON-Schema. Bei `recommend` gibt Mistral eine bereinigte englische `cleaned_query` zurueck. Diese Query wird anschliessend an `MoodPredictor` uebergeben. Deutsche User-Anfragen wie `Ich suche eine Romanze` koennen dadurch als englische Query wie `romantic movie` in die bestehende Modellpipeline laufen.

6. **Konfiguration ergaenzt**  
   In `src/settings.py` wurden `MISTRAL_API_KEY`, `MISTRAL_MODEL`, `MISTRAL_API_URL` und `MISTRAL_TIMEOUT_SECONDS` aufgenommen. `.env.template` und `README.md` dokumentieren `MISTRAL_API_KEY` und `MISTRAL_MODEL=mistral-small-latest`. Fehlt der Key oder faellt die API aus, nutzt der Bot den lokalen Fallback.

7. **Bot-Flow umgestellt**  
   Der Ablauf ist jetzt: Telegram-Nachricht -> Mistral-Entscheidung -> nur bei `action == "recommend"` Aufruf von `MoodPredictor` und `MovieRecommender`. Bei `reject`, `clarify`, `help` oder `smalltalk` antwortet der Bot direkt, ohne das Empfehlungsmodell zu belasten.

8. **Antwortformatierung mit Mistral ergaenzt**  
   Nach der Empfehlungserzeugung bekommt Mistral die fertigen Recommendations mit Titel, Score und Overview. Die Filmauswahl bleibt unveraendert aus dem Recommender. Ziel war, die starre Zeile `Here are some movies that match your request:` durch einen natuerlicheren Einstieg zu ersetzen.

9. **Formatter robuster gemacht**  
   Der erste Versuch liess Mistral die komplette finale Antwort schreiben. Dabei konnte die eigentliche Liste fehlen. Deshalb wurde der Formatter angepasst: Mistral schreibt nur noch einen kurzen Intro-Satz, die Recommendation-Liste wird deterministisch in `bot.py` angehaengt. Dadurch bleiben alle Ergebnisse sichtbar.

10. **Decision-Logging aktiviert**  
    In `bot.py` wird jede Chat-Entscheidung mit `action` und `cleaned_query` geloggt. In `main.py` wurde `logging.basicConfig(level=logging.INFO, ...)` eingerichtet, damit diese Logs in der Konsole bzw. im Docker-Log sichtbar sind. Damit laesst sich pruefen, ob Mistral deutsche Anfragen sauber in englische Modell-Queries ueberfuehrt.

Aktueller Bot-Ablauf:

```text
Telegram message
-> Mistral ChatController
   -> reject / clarify / help / smalltalk: direkte Bot-Antwort
   -> recommend: cleaned_query
-> MoodPredictor(cleaned_query)
-> MovieRecommender
-> Mistral schreibt nur Intro-Satz
-> Bot haengt Recommendation-Liste deterministisch an
```

Einordnung: Das Chat-Handling ist dadurch deutlich robuster und mehrsprachiger. Die Qualitaet der ausgewaehlten Filme haengt weiterhin vom trainierten Modell, den Labels und den Filmvektoren ab. Die Mistral-Schicht verbessert also die Chat-Steuerung und Praesentation, nicht die fachliche Qualitaet der Recommender-Ergebnisse.


## 11. Sinnvolle offene To-dos

1. **Mood-Tag-Liste kuratieren**  
   Die aktuelle Top-100-Auswahl auf echte Mood-/Tone-/Atmosphaeren-Tags reduzieren oder zumindest klar klassifizieren. Das verbessert die Bedeutung der Modellziele und ist der groesste fachliche Hebel.
   1. Umbenennung in TMDB_Tags: Wir behalten die Tags, werden uns dann aber nicht mehr nur auf "Moods" beziehen, sondern auf Genre, Mood, etc.
      1. Erhöhung auf 150 Tags.
            1. Datengrundlage muss geprüft und angepasst werden.

2. **Nach Label-Aenderungen neu trainieren**  
   Nach einer geaenderten Label-Liste sind alte Scores nicht mehr direkt vergleichbar. Danach ist eine kleinere, gezielte Experimentreihe erforderlich.

3. **Bot-Empfehlungen qualitativ evaluieren**  
   Eine feste Liste von Testprompts gegen die Top-Empfehlungen pruefen. Der Bot nutzt Wahrscheinlichkeitsvektoren per Cosine Similarity; ein hoeherer `macro_f1` garantiert deshalb nicht automatisch bessere Empfehlungen.
